<a href="https://colab.research.google.com/github/Kalianey/EpubCrawlers/blob/main/Epub_DL_From_YQK_NET_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get install -y pandoc
!pip install inlp requests beautifulsoup4

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pandoc is already the newest version (2.9.2.1-3ubuntu2).
pandoc set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 26.4 MB/s eta 0:00:00


In [2]:
# ===============================
# EPUB CRAWLER FOR YQK.NET
# ===============================
# Install required packages
!pip install inlp requests beautifulsoup4 -q

import requests
import os, re
import time
from bs4 import BeautifulSoup
from inlp.convert import chinese

# Set up workspace
try:
    os.mkdir("/content/tmp")
except:
    print("Directory exists, cleaning...")
os.chdir("/content/tmp")
os.system("rm -fr *")

# ===============================
# 1. CRAWLER FUNCTION - UPDATED FOR YQK.NET CHAPTER PAGES
# ===============================
def get_chapter(urls):
    """
    Downloads a single chapter from yqk.net
    """
    [title, art_url] = urls
    # Extract a clean ID from the URL (e.g., '68975' from .../68975.html)
    art_id = art_url[art_url.rfind("/") + 1:-5]

    print(f"Downloading: {art_id} - {title}")

    try:
        # Fetch the chapter page - NOTE ENCODING
        reg = requests.get(art_url, timeout=10)
        # CHANGED FOR YQK.NET: This site uses gb2312 encoding, not utf-8
        reg.encoding = "gb2312"
        soup = BeautifulSoup(reg.text, 'html.parser')

        # FIXED: The content is in <div class="content"> as shown in the HTML source
        content = soup.find('div', class_='content')

        if content is None:
            print(f"  ⚠️ No content with class='content' found, trying alternatives...")
            content = soup.find('div', id='content')
            if content is None:
                content = soup.find('article')
            if content is None:
                # As a last resort, get the body text
                content = soup.find('body')

        if content is None:
            print(f"  ❌ Could not find content container on {art_url}")
            text = f"# {title}\n\n[Content not found on this page]"
        else:
            # Remove unwanted elements - MORE SPECIFIC FOR YQK.NET
            # Remove advertisement divs, scripts, navigation, etc.
            for unwanted in content.find_all(['div', 'script', 'nav', 'footer', 'hr', 'iframe', 'ins', 'table', 'dl', 'style', 'form', 'input', 'button']):
                unwanted.decompose()

            # Also remove the navigation links at the bottom
            for nav in content.find_all('p', class_='move'):
                nav.decompose()

            # Remove the mygg div (advertisement container)
            for mygg in content.find_all('div', class_='mygg'):
                mygg.decompose()

            # Remove the javascript block at the bottom
            for script_block in content.find_all('script'):
                script_block.decompose()

            # Get text and clean it
            raw_text = content.get_text()

            # Clean up the text
            lines = []
            for line in raw_text.split('\n'):
                line = line.strip()
                # Skip empty lines and certain promotional text
                if not line or '言情库' in line or 'yqk.net' in line or '加入收藏' in line:
                    continue
                # Skip page navigation lines
                if '上一章' in line or '下一章' in line or '快捷键' in line or '返回目录' in line:
                    continue
                # Skip advertisement lines
                if 'ADVERTISEMENT' in line or '广告' in line or 'pagead2.googlesyndication.com' in line:
                    continue
                # Skip the copyright/footer text
                if '本站所有小说为转载作品' in line or '执行时间：' in line:
                    continue
                lines.append(line)

            # Join into paragraphs
            cleaned_text = "\n\n".join(lines)

            # Add chapter title
            text = f"# {title}\n\n{cleaned_text}\n\n"

            # Optional: Convert to Traditional Chinese
            # text = chinese.s2t(text)

        # Save as .txt file
        with open(f"{art_id}.txt", mode="w", encoding="utf-8") as f:
            f.write(text)

        # Small delay to be polite to the server
        time.sleep(0.5)

    except Exception as e:
        print(f"  ❌ Error downloading {art_id}: {e}")

# ===============================
# 2. CONFIGURATION FOR THE NEW SITE
# ===============================
# === CHANGE THESE VALUES ===
# CHANGED FOR YQK.NET: Using the new URL structure
BOOK_URL = "https://www.yqk.net/yanqing/bolixin/index.html"
# Extracted from the page title and metadata
BOOK_TITLE = "玻璃心"  # "Glass Heart"
BOOK_AUTHOR = "镜水"  # "Jing Shui"
MAKE_EPUB = True  # Set to False if you only want text files
# ===========================

# Create metadata file for EPUB
YAML = f'''---
title: {BOOK_TITLE}
author: {BOOK_AUTHOR}
language: zh-Hans
---'''
with open("title.txt", mode="w", encoding='utf-8') as f:
    f.write(YAML)

# ===============================
# 3. FETCH CHAPTER LIST - COMPLETELY REVISED FOR YQK.NET
# ===============================
BASE_SITE = "https://www.yqk.net"
print(f"📚 Fetching table of contents from: {BOOK_URL}")

try:
    # CHANGED FOR YQK.NET: Use gb2312 encoding for the main page too
    reg = requests.get(BOOK_URL, timeout=10)
    reg.encoding = "gb2312"
    soup = BeautifulSoup(reg.text, 'html.parser')

    # CHANGED FOR YQK.NET: Find the chapter list in <dl class="chapter">
    # Looking at the HTML source, chapters are in: <dl class="chapter"> ... <dd><a href="...">
    chapter_container = soup.find('dl', class_='chapter')

    if chapter_container is None:
        print("❌ Could not find chapter container with class='chapter'.")
        # Try alternative selectors
        chapter_container = soup.find('div', class_='chapter-list') or \
                          soup.find('div', id='chapter-list') or \
                          soup.find_all('dl')[-1]  # Last dl on page

    if chapter_container:
        # Get all chapter links within the container
        articles = chapter_container.find_all('a')
        print(f"Found {len(articles)} chapter links in the container")
    else:
        print("❌ No chapter container found. Trying to find all chapter links...")
        # Fallback: Find all links that look like chapter links
        articles = []
        all_links = soup.find_all('a', href=re.compile(r'\d+\.html'))
        for link in all_links:
            href = link.get('href', '')
            if href and re.match(r'^\d+\.html$', href):
                articles.append(link)
        print(f"Found {len(articles)} chapter links via regex fallback")

    chapter_links = []
    for link_tag in articles:
        href = link_tag.get("href", "")
        if not href:
            continue

        # Skip non-chapter links
        if not href.endswith(".html"):
            continue

        # Check if it's a numeric chapter link (like 68975.html)
        if not re.match(r'^\d+\.html$', href):
            continue

        # Get chapter title
        chapter_title = link_tag.get_text().strip()
        if not chapter_title:
            # Use the filename as title
            chapter_title = f"Chapter {href[:-5]}"

        # Construct full URL
        if href.startswith('http'):
            full_url = href
        elif href.startswith('/'):
            full_url = f"{BASE_SITE}{href}"
        else:
            # Relative path like "68975.html"
            # From the HTML, the index is at /yanqing/bolixin/index.html
            # Chapter links are like "68975.html" (relative to current directory)
            # So the full URL should be: https://www.yqk.net/yanqing/bolixin/68975.html
            full_url = f"https://www.yqk.net/yanqing/bolixin/{href}"

        chapter_links.append([chapter_title, full_url])

    # Remove duplicates while preserving order
    seen = set()
    unique_chapter_links = []
    for item in chapter_links:
        key = item[1]  # Use URL as unique key
        if key not in seen:
            seen.add(key)
            unique_chapter_links.append(item)

    chapter_links = unique_chapter_links

    print(f"✅ Found {len(chapter_links)} unique chapters.")

except Exception as e:
    print(f"❌ Error fetching table of contents: {e}")
    import traceback
    traceback.print_exc()
    chapter_links = []

# Show first few chapters for verification
if chapter_links:
    print("\nFirst 5 chapters to download:")
    for i, (title, url) in enumerate(chapter_links[:5]):
        print(f"  {i+1}. {title} -> {url}")
    if len(chapter_links) > 5:
        print(f"  ... and {len(chapter_links)-5} more chapters")
else:
    print("❌ No chapters found. Check the BOOK_URL or website structure.")

# ===============================
# 4. DOWNLOAD ALL CHAPTERS
# ===============================
if chapter_links:
    print(f"\n⏬ Downloading {len(chapter_links)} chapters...")
    for i, chapter in enumerate(chapter_links, 1):
        print(f"\n[{i}/{len(chapter_links)}] ", end="")
        get_chapter(chapter)
else:
    print("❌ Cannot download - no chapters found.")

# ===============================
# 5. CREATE EPUB
# ===============================
if MAKE_EPUB and chapter_links:
    # Get list of all downloaded .txt files
    txt_files = []
    for link in chapter_links:
        filename = link[1][link[1].rfind("/") + 1:-5] + ".txt"
        if os.path.exists(filename):
            txt_files.append(filename)

    if txt_files:
        print(f"\n📖 Creating EPUB from {len(txt_files)} chapters...")
        files_string = " ".join(txt_files)

        # Run pandoc to create EPUB
        os.system(f'pandoc -o "../{BOOK_TITLE}.epub" title.txt {files_string}')

        if os.path.exists(f"../{BOOK_TITLE}.epub"):
            print(f"✅ EPUB created successfully: {BOOK_TITLE}.epub")
            print("📥 Download should start automatically...")
            from google.colab import files
            files.download(f'../{BOOK_TITLE}.epub')
        else:
            print("❌ EPUB creation failed.")
    else:
        print("❌ No chapter files found to create EPUB.")
else:
    print("\n📝 Creating combined text file...")
    with open(f"../{BOOK_TITLE}.txt", "w", encoding='utf-8') as outfile:
        outfile.write(f"{BOOK_TITLE}\n")
        outfile.write(f"By {BOOK_AUTHOR}\n")
        outfile.write("=" * 50 + "\n\n")

        for link in chapter_links:
            filename = link[1][link[1].rfind("/") + 1:-5] + ".txt"
            if os.path.exists(filename):
                with open(filename, "r", encoding='utf-8') as infile:
                    outfile.write(infile.read())
                    outfile.write("\n\n")
            else:
                print(f"  Missing: {filename}")

    from google.colab import files
    files.download(f'../{BOOK_TITLE}.txt')

print("\n✨ Done!")

Directory exists, cleaning...
📚 Fetching table of contents from: https://www.yqk.net/yanqing/bolixin/index.html
Found 13 chapter links in the container
✅ Found 13 unique chapters.

First 5 chapters to download:
  1. 楔子--心动 -> https://www.yqk.net/yanqing/bolixin/68975.html
  2. 第一章 -> https://www.yqk.net/yanqing/bolixin/68976.html
  3. 第二章 -> https://www.yqk.net/yanqing/bolixin/68977.html
  4. 第三章 -> https://www.yqk.net/yanqing/bolixin/68978.html
  5. 第四章 -> https://www.yqk.net/yanqing/bolixin/68979.html
  ... and 8 more chapters

⏬ Downloading 13 chapters...

[1/13] Downloading: 68975 - 楔子--心动

[2/13] Downloading: 68976 - 第一章

[3/13] Downloading: 68977 - 第二章

[4/13] Downloading: 68978 - 第三章

[5/13] Downloading: 68979 - 第四章

[6/13] Downloading: 68980 - 第五章

[7/13] Downloading: 68981 - 第六章

[8/13] Downloading: 68982 - 第七章

[9/13] Downloading: 68983 - 第八章

[10/13] Downloading: 68984 - 第九章

[11/13] Downloading: 68985 - 尾声

[12/13] Downloading: 68986 - 在那之后

[13/13] Downloading: 68987 - 后记


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✨ Done!
